In [ ]:
import knmi
import pandas as pd

start = pd.Timestamp("20200901", tz="Europe/Brussels")
end = pd.Timestamp("20261031", tz="Europe/Brussels")
variables = ["PRCP"]

In [ ]:
def filter_months(df: pd.DataFrame, months: list[int]):
    return df[df.Date.dt.month.isin(months)]

def filter_day (df, col, start, end):
    df[col] = pd.to_datetime(df[col])
    df.set_index(col, inplace=True)
    day_df = df.between_time(start, end)
    return day_df

In [ ]:
df = (
    knmi.get_hour_data_dataframe(
        stations=[260], 
        start=start, 
        end=end, 
        variables=variables
    )
    .reset_index()
    .pipe(lambda df: df.replace({"RH": {-1: 0}}))
    .rename(columns={
        "DR":"Duration prec (in 0.1 hour)",
        "RH":"Daily precipitation amount (in 0.1 mm)",
        "YYYYMMDD_HH": "Date",
        "STN":"Station"})
    .pipe(filter_months, [9, 10, 11])
    .pipe(filter_day, "Date", "06:00", "18:00")
    .pipe(lambda df: df.dropna())
)
df

In [ ]:
df.reset_index().to_csv("filtered_data.csv", index=False)